<a href="https://colab.research.google.com/github/HugoCrainich/Statistical-Machine-Learning/blob/main/Lab%2013/Lab_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# Step 1: Ingestion and Naive Model
url = 'https://raw.githubusercontent.com/HugoCrainich/Statistical-Machine-Learning/refs/heads/main/data/Zillow_California_2026_Hedonic.csv'
df = pd.read_csv(url)

naive_model = smf.ols('Sale_Price ~ Property_Age', data=df).fit()
print(naive_model.summary())
print("\nNaive Age Coefficient:", naive_model.params['Property_Age'])

# Step 2: The Multivariate Model
multi_model = smf.ols('Sale_Price ~ Property_Age + Distance_to_Tech_Hub', data=df).fit()
print(multi_model.summary())
print("\nMultivariate Age Coefficient:", multi_model.params['Property_Age'])

# Step 3: FWL Theorem Manual Proof
# 3a: Partial out distance from Price
res_y_model = smf.ols('Sale_Price ~ Distance_to_Tech_Hub', data=df).fit()
df['Price_Residuals'] = res_y_model.resid

# 3b: Partial out distance from Age
res_x_model = smf.ols('Property_Age ~ Distance_to_Tech_Hub', data=df).fit()
df['Age_Residuals'] = res_x_model.resid

# 3c: Regress Residuals on Residuals (-1 removes the intercept for exact mathematical matching)
fwl_model = smf.ols('Price_Residuals ~ Age_Residuals - 1', data=df).fit()
print("\nFWL Isolated Age Coefficient:", fwl_model.params['Age_Residuals'])


                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     3105.
Date:                Wed, 18 Mar 2026   Prob (F-statistic):          1.26e-308
Time:                        21:27:03   Log-Likelihood:                -12818.
No. Observations:                1000   AIC:                         2.564e+04
Df Residuals:                     998   BIC:                         2.565e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     3.013e+05   7218.570     41.742   

In [6]:
"""
Multivariate OLS Regression: 3D Scatter + Fitted Hyperplane
Predictors : Property_Age (years), Distance_to_Tech_Hub (miles)
Response   : Sale_Price (USD)
Libraries  : numpy, pandas, statsmodels, plotly.graph_objects
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go

# ── 1. SIMULATE DATA ──────────────────────────────────────────────────────────
# Replace this block with your own DataFrame if you already have the data.
rng = np.random.default_rng(42)
n   = 200

Property_Age        = rng.uniform(0,   50,  n)   # years since construction
Distance_to_Tech_Hub = rng.uniform(0,   30,  n)   # miles from nearest tech hub
noise               = rng.normal(0, 40_000, n)

# True DGP: older and farther-from-hub → lower sale price
Sale_Price = (
    750_000
    - 4_500  * Property_Age
    - 9_000  * Distance_to_Tech_Hub
    + noise
)

df = pd.DataFrame({
    "Sale_Price":           Sale_Price,
    "Property_Age":         Property_Age,
    "Distance_to_Tech_Hub": Distance_to_Tech_Hub,
})

# ── 2. FIT OLS WITH statsmodels ───────────────────────────────────────────────
# sm.add_constant prepends a column of 1s so statsmodels estimates β₀ (intercept).
X = sm.add_constant(df[["Property_Age", "Distance_to_Tech_Hub"]])
y = df["Sale_Price"]

model  = sm.OLS(y, X)
result = model.fit()

print(result.summary())

# ── 3. EXTRACT COEFFICIENTS ───────────────────────────────────────────────────
# result.params is a pandas Series indexed by column name.
# Accessing by name (not position) is safer if column order ever changes.
beta0 = result.params["const"]                  # intercept β₀
beta1 = result.params["Property_Age"]           # slope for Property_Age (β₁)
beta2 = result.params["Distance_to_Tech_Hub"]   # slope for Distance_to_Tech_Hub (β₂)

print(f"\nExtracted coefficients:")
print(f"  β₀ (Intercept)            = {beta0:,.2f}")
print(f"  β₁ (Property_Age)         = {beta1:,.2f}")
print(f"  β₂ (Distance_to_Tech_Hub) = {beta2:,.2f}")

# ── 4. BUILD THE MESHGRID FOR THE REGRESSION PLANE ────────────────────────────
# np.linspace creates 50 evenly-spaced values spanning each predictor's range.
# This gives us the X and Y "axes" of the grid.
age_range  = np.linspace(df["Property_Age"].min(),          df["Property_Age"].max(),          50)
dist_range = np.linspace(df["Distance_to_Tech_Hub"].min(),  df["Distance_to_Tech_Hub"].max(),  50)

# np.meshgrid converts the two 1D arrays into two 2D arrays (50×50 each).
# age_grid[i, j]  = the Property_Age  value at grid cell (i, j)
# dist_grid[i, j] = the Distance      value at grid cell (i, j)
# Together they represent every (age, dist) combination on the grid surface.
age_grid, dist_grid = np.meshgrid(age_range, dist_range)

# Apply the OLS equation: Ŷ = β₀ + β₁·X₁ + β₂·X₂
# Because age_grid and dist_grid are 2D NumPy arrays, this vectorised operation
# returns a 2D predicted-price surface (price_grid) of the same shape (50×50).
price_grid = beta0 + beta1 * age_grid + beta2 * dist_grid

# ── 5. BUILD THE PLOTLY FIGURE ────────────────────────────────────────────────
fig = go.Figure()

# — 5a. SCATTER: actual observed data points ——————————————————————————————————
fig.add_trace(go.Scatter3d(
    x    = df["Property_Age"],
    y    = df["Distance_to_Tech_Hub"],
    z    = df["Sale_Price"],
    mode = "markers",
    marker = dict(
        size        = 4,
        color       = df["Sale_Price"],        # colour encodes sale price magnitude
        colorscale  = "Viridis",
        opacity     = 0.80,
        colorbar    = dict(title="Sale Price ($)", x=1.02),
    ),
    name       = "Observed data",
    hovertemplate = (
        "<b>Property Age:</b> %{x:.1f} yrs<br>"
        "<b>Distance:</b> %{y:.1f} mi<br>"
        "<b>Sale Price:</b> $%{z:,.0f}<extra></extra>"
    ),
))

# — 5b. SURFACE: the fitted OLS regression hyperplane ————————————————————————
# go.Surface takes 2D arrays directly — age_grid and dist_grid become the x/y
# coordinates; price_grid (the 50×50 predicted values) becomes z.
# showscale=False hides a second colour bar (the scatter already has one).
fig.add_trace(go.Surface(
    x          = age_grid,
    y          = dist_grid,
    z          = price_grid,
    colorscale = "Blues",
    opacity    = 0.55,
    showscale  = False,
    name       = "Fitted hyperplane",
    hovertemplate = (
        "<b>Age:</b> %{x:.1f} yrs<br>"
        "<b>Distance:</b> %{y:.1f} mi<br>"
        "<b>Predicted Price:</b> $%{z:,.0f}<extra></extra>"
    ),
))

# ── 6. LAYOUT ─────────────────────────────────────────────────────────────────
fig.update_layout(
    title = dict(
        text = (
            f"OLS Hyperplane: Sale Price ~ Property_Age + Distance_to_Tech_Hub<br>"
            f"<sup>β₀={beta0:,.0f}  |  β₁(Age)={beta1:,.0f}  |  β₂(Dist)={beta2:,.0f}</sup>"
        ),
        x    = 0.5,
        font = dict(size=14),
    ),
    scene = dict(
        xaxis = dict(title="Property Age (yrs)"),
        yaxis = dict(title="Distance to Tech Hub (mi)"),
        zaxis = dict(title="Sale Price ($)"),
        camera = dict(eye=dict(x=1.6, y=-1.6, z=1.0)),  # initial viewing angle
    ),
    legend    = dict(x=0.02, y=0.98),
    margin    = dict(l=0, r=0, t=80, b=0),
    width     = 900,
    height    = 700,
    template  = "plotly_white",
)

fig.show()
# fig.write_html("ols_3d_hyperplane.html")   # ← uncomment to save standalone HTML

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.858
Model:                            OLS   Adj. R-squared:                  0.857
Method:                 Least Squares   F-statistic:                     596.1
Date:                Wed, 18 Mar 2026   Prob (F-statistic):           2.78e-84
Time:                        21:27:03   Log-Likelihood:                -2400.7
No. Observations:                 200   AIC:                             4807.
Df Residuals:                     197   BIC:                             4817.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                 7.421e+05 